In [ ]:
%load_ext autoreload
%autoreload 2
# automatically pick up changes made in src file

# LEDGAR: preprocessing (run ONCE)
Cleans, lemmatises and builds TF-IDF, then saves everything so the modelling notebooks never redo it.

**Run top to bottom once.** Re-run only if you change cleaning, lemmatising, stop words or TF-IDF settings.

**Saves**
- `data/processed/proc_*.parquet` - lemmatised clauses (section 3c)
- `data/features/x_train.npz`, `x_val.npz`, `x_test.npz` - TF-IDF matrices
- `data/features/vectorizer.joblib` - the fitted TF-IDF (vocabulary + IDF)

## 1. Imports
All src files are imported here, once.

In [ ]:
from pathlib import Path
import sys; sys.path.append('..')     # lets the notebook find the src folder

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

from src.clean import clean_text # cleaning function
from src.lemmetizer import Lemmatizer # spaCy lemmatiser class
from src.tf_idf import tfid # builds the TF-IDF vectoriser
from src.multi_class_lr_softmax import MultiClassLogisticRegression # model class
from src.classification_error_analysis import ErrorAnalysis # error analysis class

## 2. Load the EDA output
- `train`: de-duplicated, lowercased, with `word_count` (saved by `eda.ipynb`)
- validation and test were not changed in the EDA, so they are read from `data/raw` in the cleaning step

In [ ]:
train = pd.read_parquet("../data/interim/train_eda.parquet")
print("loaded train:", train.shape)   # expect (59933, 5)
train.head()

## 3. Text preprocessing
Bag of words, TF-IDF, Word2Vec all need CLEAN text first.
- text preprocessing: lowercasing, remove special characters, tokenising, stop words, lemmatisation
- Bag of words - turning text into word counts - TF-IDF works on this foundation
- Word2Vec / sentence embeddings - used downstream

**Steps**
1. Read a sample of raw clauses. Note every odd pattern: numbers, money, (ii), punctuation and so on.
2. Cleaning function with `re` (called from `src/clean.py`, so it is reproducible). Takes one clause and returns a cleaner version:
    - lowercase
    - money becomes moneytoken
    - numbers become numtoken
    - remove bracketed list markers like (a) and (ii)
    - remove remaining punctuation and symbols
    - collapse extra spaces
3. Lemmatise with spaCy. Feed it the cleaned clauses, and get back each clause with words in base form. spaCy tokenises internally to do this.
4. Save it. Add the result as a column and save to `data/processed`, so the slow lemmatising only runs once.
5. Find legal stop words (section 4).
6. TF-IDF: give TfidfVectorizer the lemmatised column and the stop-word list. It tokenises, removes stop words and builds the numbers (section 5).

### 3a. Look at raw clauses

In [ ]:
train.text
# different types of termanology to consider when cleaning: dates written in long format, money, numbers, section refrences, numbering in lists, addresses, phone numbers, emails, urls

### 3b. Clean
Test the cleaning function on one example first, then apply it to train, validation and test.

In [ ]:
# test the cleaning function first:
testing_text = 'Notices: (a) send to legal@acme.com within 30 days; (ii) pay $1,250,000.00 per Section 5.2 of the Agreement.'
result = clean_text(testing_text)
print(result)

In [ ]:
# clean all files: train, validation , test
splits = {
    "train": train,
    "validation": pd.read_parquet("../data/raw/validation.parquet"),
    "test": pd.read_parquet("../data/raw/test.parquet"),
}

for name, df in splits.items():
    df["clean_text"] = df["text"].apply(clean_text)
    print(name, df.shape)

### 3c. Tokenise and lemmatise (spaCy)
- spaCy's small English model is a chain of steps: tokenizer → tagger → attribute_ruler → lemmatizer → parser → ner
- lemmatising does not need parser and ner, so they are turned off (much faster)
- the `Lemmatizer` class (`src/lemmetizer.py`) loads spaCy ONCE; the model lives in `lem.nlp`
- one-time install, in the terminal (not here): `python -m spacy download en_core_web_sm`

In [ ]:
lem = Lemmatizer()   # loads spaCy once

# test the lemmatiser on the cleaned example (it tokenises too)
doc = lem.nlp(result)
print([token.lemma_ for token in doc])

In [ ]:
processed = Path("../data/processed")
processed.mkdir(parents=True, exist_ok=True)

for name, df in splits.items(): # .items (loop over dictinoary) gives both the key and value, i.e. name and df
    df = lem.lemmatize_clauses(df) # tokenise + lemmatise -> new column: lemmatized_text
    df.to_parquet(processed / f"proc_{name}.parquet", index=False) # save
    print(name, df.shape)

### 3d. Sanity check after preprocessing
- cleaning removes apostrophes
- the lemmatiser can then mangle lone letters: won't → won t → win
- so check how many informal contractions are in the text column

In [ ]:
train[train.text.str.contains("'t")] #  no rows therefore no contractions have been used, all meaning remain the same in setnences of contractions, e.g. "don't" = "do not", "can't" = "cannot", "won't" = "will not"

## 4. Stop words
Start from scikit-learn's English stop-word list, then decide which words to keep and which legal words to add.

In [ ]:
# extract english stop words to find out which need to be made exempt from the list:
print(f"English stop words: {len(ENGLISH_STOP_WORDS)}")
sorted_stop_words = sorted(list(ENGLISH_STOP_WORDS))

# Write each word to a new line in a text file
with open('sklearn_english_stop_words.txt', 'w', encoding='utf-8') as file:
    for word in sorted_stop_words:
        file.write(f"{word}\n")

- Check stop words - if they turn up mostly in specific clause types, they help separate classes = keep
- Words such as amount, due, full, against, first and per were left in the stop-word list. They are only moderately concentrated in some classes; keeping them can be tested later on validation

In [ ]:
# share of clauses containing each word, and which clause types they appear in
# \b = whole word only (otherwise "per" also matches perform, period, property)
for word in ("amongst", "per", "full", "against", "without", "within", "between", "among", "first", "except", "otherwise", "bill"):
    has_word = train.text.str.contains(rf"\b{word}\b")
    print(f"Clauses containing '{word}': {has_word.mean()*100:.2f}%")
    print(train[has_word]['label_name'].value_counts(normalize=True).round(2).head(5) * 100, "\n")

Words exempt from the stop-word list - removing them would change the meaning of clauses, and so the classification:
- **Negations** (they flip meaning): no · not · nor · never · neither · none · nothing · nobody · without · cannot
- **Clause-name words**: further (from Further Assurances) · interest (from Interests)

In [ ]:
stop_words_set = set(ENGLISH_STOP_WORDS)

# KEEP in text (remove them from the stop-word list)
exempt_words = {'no', 'not', 'nor', 'never', 'neither', 'none', 'nothing',
                'nobody', 'without', 'cannot', 'further', 'interest'}

# Common legal words to REMOVE from the text (add them to the stop-word list)
legal_words = {'shall', 'party', 'agreement', 'hereof', 'hereunder', 'herein'}

# Final list: built-in + legal words - exemptions
stop_words_set = (stop_words_set | legal_words) - exempt_words

print(type(stop_words_set), len(stop_words_set))   # must stay a set of WORDS - don't convert to str

## 5. TF-IDF
- Validation and test play the role of future data the model has never seen. Their whole job is to tell you how the model will do in real life.
- train: `fit_transform` (learn the vocabulary + IDF, then convert)
- validation and test: `transform` ONLY (no leakage)

In [ ]:
# read in the lemmatised files:
proc_train = pd.read_parquet(processed / "proc_train.parquet")
proc_val = pd.read_parquet(processed / "proc_validation.parquet")
proc_test = pd.read_parquet(processed / "proc_test.parquet")
proc_train.columns

In [ ]:
vectorizer = tfid(list(stop_words_set))

# fit and transform the train dataset:
x_train = vectorizer.fit_transform(proc_train.lemmatized_text)

# transform the learned vectorizer on validation and test - no leakage
x_val = vectorizer.transform(proc_val.lemmatized_text)
x_test = vectorizer.transform(proc_test.lemmatized_text)

feature_names = vectorizer.get_feature_names_out()   # the column (word) names
print("x_train:", x_train.shape, "| non-zeros:", x_train.nnz)

### Observe the sparse matrix

In [ ]:
i = 0                                   # row number of the clause to look at
row = x_train[i]
words = feature_names[row.indices]      # the columns this clause actually uses
scores = row.data                       # their TF-IDF scores

print(proc_train["lemmatized_text"].iloc[i])
print(pd.Series(scores, index=words).sort_values(ascending=False).head(15))
pd.DataFrame(x_train[:5, :10].toarray(), columns=feature_names[:10])

The TF-IDF matrix has ~95,000 features, but each clause uses only about 73 of them on average; the matrix is over 99.9% empty, so it is stored as a sparse matrix.

(Update the numbers after rerunning with the stop-word fix.)

## 6. Save the features
Saves the TF-IDF matrices and the fitted vectoriser. The modelling notebooks load these instead of rebuilding them.

In [ ]:
from scipy import sparse
import joblib

features = Path("../data/features")
features.mkdir(parents=True, exist_ok=True)

sparse.save_npz(features / "x_train.npz", x_train)  # sparse format: small file, fast to load
sparse.save_npz(features / "x_val.npz", x_val)
sparse.save_npz(features / "x_test.npz", x_test)
joblib.dump(vectorizer, features / "vectorizer.joblib") # vocabulary + IDF, to transform new clauses later

print("saved:", x_train.shape, x_val.shape, x_test.shape)